# 32.01 Условные критерии $L_{min}$ боковых сборок

> **Статус:** канонический синтетический расчёт трёх семейств критериев на
> изготовленном ряду 50–140 мм. Принятых научных порогов пока нет, поэтому
> реальный $L_{min}$ экспериментов 2–3 не определён.

Notebook не читает данные добровольцев и не выбирает оптимальную пару. Он
показывает, как ответ меняется при изменении условного порога и эффективной
толщины $h$ в идеальной плоской двуслойной модели.


## Происхождение и исправленная постановка

Источники — критерии из старых документов `04`, `05`, `12`, формализованные в
`31.00` §6 и масштабно проверенные в `30.05`. Исторически использовались
числа $c=3$, $\eta_{min}=1\%$, $q_2^*=0{,}1\ldots0{,}3$ и пороги
$S^*,\mathrm{FoM}^*$ без экспериментального обоснования. Они не переносятся
как принятые требования.

Исправлены три смешения:

1. **Модельный минимум и изготовленный ряд.** Ниже находится первый прошедший
   кандидат только среди $L=50,60,\ldots,140$ мм. Это не непрерывный минимум и
   не рекомендация изготовить такой размер.
2. **$L_{min}$ и $L_{max}$.** Нижняя граница информативности не доказывает
   применимость плоской модели. Верхнее допустимое множество задаётся отдельно
   субъектным КТ/FEM-критерием `20.12`; универсального $L_{max}$ нет.
3. **Односборочная чувствительность и идентифицируемость пары.** Условие по
   $S_{\rho_2}$ и FoM является только предварительным фильтром одной сборки.
   Разделение $\rho_1/\rho_2$ требует пары, SVD/ковариации и проверки данных в
   `31.03` и `32.02`.


## Три семейства критериев

Для синтетического дыхательного размаха

$$
\Delta Z_{breath}(L,h)=Z(\rho_1,\rho_2^{in},h;L)-
Z(\rho_1,\rho_2^{ex},h;L)
$$

используются:

1. **Абсолютная/относительная измеримость**

   $$|\Delta Z_{breath}|\ge T_Z,\qquad
   |\Delta Z_{breath}/Z_{mid}|\ge T_{rel}.$$

   Для реального критерия $T_Z=c\,u(\widehat{\Delta Z})$ должен строиться по
   неопределённости конкретного оценивателя и обработки. Реестр необходимых
   компонент дан в `31.04`; паспортное разрешение прибора его не заменяет.

2. **Нормированная доля межслойного контраста**

   $$q_2=\frac{\rho_a-\rho_1}{\rho_2-\rho_1}\ge q_2^*.$$

   $q_2$ — модельный показатель кажущегося сопротивления, а не доля тока,
   буквально прошедшего через лёгкое. Критерий не использует дыхательный
   размах и сам по себе не проверяет его измеримость.

3. **Односборочный фильтр чувствительности**

   $$|S_{\rho_2}|\ge S^*,\qquad
   \mathrm{FoM}=|S_{\rho_2}|/|S_h|\ge F^*.$$

   FoM описывает только локальный перенос ошибки $h$ при фиксированных
   остальных параметрах. Он не включает gain/offset, контакт, переклейку,
   последовательность записей, ошибку геометрии и расхождение модели.

Все пороги в исполняемой части помечены `illustrative_unvalidated` и нужны
только для проверки логики алгоритма.


In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
model_paths = sorted({
    (candidate / "two_layer_model.py").resolve()
    for candidate in candidates
    if (candidate / "two_layer_model.py").is_file()
})
if len(model_paths) != 1:
    raise RuntimeError(f"expected one canonical two_layer_model.py, found: {model_paths}")

model_path = model_paths[0]
sys.path.insert(0, str(model_path.parent))
import two_layer_model as tlm

if Path(tlm.__file__).resolve() != model_path:
    raise RuntimeError(f"imported non-canonical model: {tlm.__file__}")

print(f"Python {platform.python_version()}; NumPy {np.__version__}")
print(f"Модель: {model_path}")


In [ ]:
# Синтетические параметры; не оценки добровольцев.
RHO1 = 5.0
RHO2_MID = 20.0
RHO2_EXHALE = 15.0
RHO2_INHALE = 25.0
BETA = 0.5
H_SCENARIOS_M = np.array([0.010, 0.020, 0.030, 0.040])
MANUFACTURED_SIZES_M = np.arange(0.050, 0.141, 0.010)
MODEL_VALIDITY = "unverified_without_subject_specific_CT_FEM_Lmax"

THRESHOLDS = {
    "absolute_ohm": {
        "status": "illustrative_unvalidated",
        "values": (1.0, 5.0, 10.0),
        "source": "арифметические сценарии; не u(DeltaZ) прибора",
    },
    "relative": {
        "status": "illustrative_unvalidated",
        "values": (0.01, 0.05, 0.10),
        "source": "арифметические сценарии; не принятая точность",
    },
    "q2": {
        "status": "illustrative_unvalidated",
        "values": (0.10, 0.20, 0.30),
        "source": "исторический планировочный диапазон; не валидирован",
    },
    "sensitivity_fom": {
        "status": "illustrative_unvalidated",
        "values": ((0.10, 0.50), (0.20, 1.00), (0.30, 1.00)),
        "source": "сценарии one-size prefilter; не критерий пары",
    },
}

print("status=illustrative_unvalidated")
print(f"model_validity={MODEL_VALIDITY}")
print("h, мм:", [int(round(h * 1000)) for h in H_SCENARIOS_M])
print("изготовленный ряд, мм:", [int(round(size * 1000)) for size in MANUFACTURED_SIZES_M])
for name, specification in THRESHOLDS.items():
    print(f"{name}: {specification['values']} [{specification['status']}] — {specification['source']}")


In [ ]:
def metrics_for(size_m, h_m):
    a, b = tlm.geometry_from_size(size_m, BETA)
    mid = tlm.evaluate(RHO1, RHO2_MID, h_m, a, b)
    z_exhale = tlm.transfer_impedance(RHO1, RHO2_EXHALE, h_m, a, b)
    z_inhale = tlm.transfer_impedance(RHO1, RHO2_INHALE, h_m, a, b)
    delta_z = z_inhale - z_exhale
    apparent = tlm.apparent_resistivity(mid.z, a, b)
    q2 = (apparent - RHO1) / (RHO2_MID - RHO1)
    s_rho2 = RHO2_MID * mid.d_rho2 / mid.z
    s_h = h_m * mid.d_h / mid.z
    fom = abs(s_rho2) / abs(s_h) if s_h != 0 else np.inf
    return {
        "z_mid": mid.z,
        "delta_z": delta_z,
        "relative": abs(delta_z / mid.z),
        "q2": q2,
        "s_rho2": s_rho2,
        "s_h": s_h,
        "fom": fom,
    }


def first_passing_size(predicate):
    for size_m in MANUFACTURED_SIZES_M:
        if predicate(float(size_m)):
            return float(size_m)
    return None


def format_candidate(size_m):
    return "нет кандидата в 50–140 мм" if size_m is None else f"{size_m*1000:.0f} мм"


# Численные и физические самопроверки в пределах выбранного синтетического диапазона.
for h_m in H_SCENARIOS_M:
    for size_m in MANUFACTURED_SIZES_M:
        row = metrics_for(float(size_m), float(h_m))
        assert np.isfinite(list(row.values())).all()
        assert -1e-12 <= row["q2"] <= 1.0 + 1e-12
        assert row["relative"] >= 0 and row["fom"] >= 0

reference_50 = metrics_for(0.050, 0.020)["delta_z"]
reference_140 = metrics_for(0.140, 0.020)["delta_z"]
np.testing.assert_allclose(reference_50, 3.9103281312, rtol=1e-7, atol=1e-9)
np.testing.assert_allclose(reference_140, 9.2724544689, rtol=1e-7, atol=1e-9)
assert np.isclose(first_passing_size(lambda size: size >= 0.090), 0.090)
assert first_passing_size(lambda size: False) is None
print("Самопроверки метрик и дискретного поиска: пройдены")


In [ ]:
h_reference = 0.020
print("Синтетические метрики при h=20 мм (не данные добровольца):")
print("L, мм | DeltaZ, Ом | |DeltaZ/Zmid| | q2 | S_rho2 | FoM")
for size_m in MANUFACTURED_SIZES_M:
    row = metrics_for(float(size_m), h_reference)
    print(
        f"{size_m*1000:5.0f} | {row['delta_z']:10.4f} | {row['relative']:13.4f} | "
        f"{row['q2']:.4f} | {row['s_rho2']:.4f} | {row['fom']:.4f}"
    )


In [ ]:
def print_scenario_table(title, threshold_values, candidate_for):
    print(f"\n{title}")
    print("порог | " + " | ".join(f"h={h*1000:.0f} мм" for h in H_SCENARIOS_M))
    for threshold in threshold_values:
        candidates = [candidate_for(float(h), threshold) for h in H_SCENARIOS_M]
        print(f"{threshold!s:>12} | " + " | ".join(format_candidate(value) for value in candidates))


print_scenario_table(
    "Абсолютный порог |DeltaZ| >= T_Z, Ом [illustrative_unvalidated]",
    THRESHOLDS["absolute_ohm"]["values"],
    lambda h, threshold: first_passing_size(
        lambda size: abs(metrics_for(size, h)["delta_z"]) >= threshold
    ),
)

print_scenario_table(
    "Относительный порог |DeltaZ/Zmid| >= T_rel [illustrative_unvalidated]",
    THRESHOLDS["relative"]["values"],
    lambda h, threshold: first_passing_size(
        lambda size: metrics_for(size, h)["relative"] >= threshold
    ),
)

print_scenario_table(
    "Модельный порог q2 >= q2* [illustrative_unvalidated]",
    THRESHOLDS["q2"]["values"],
    lambda h, threshold: first_passing_size(
        lambda size: metrics_for(size, h)["q2"] >= threshold
    ),
)

print_scenario_table(
    "One-size prefilter (S*, FoM*) [illustrative_unvalidated]",
    THRESHOLDS["sensitivity_fom"]["values"],
    lambda h, thresholds: first_passing_size(
        lambda size: (
            abs(metrics_for(size, h)["s_rho2"]) >= thresholds[0]
            and metrics_for(size, h)["fom"] >= thresholds[1]
        )
    ),
)


## Как читать результат

- Строка таблицы отвечает только на вопрос: «какой **первый изготовленный**
  размер прошёл данный условный порог в данной синтетической точке?»
- Ответ `нет кандидата в 50–140 мм` не означает, что нужно увеличивать сборку:
  сначала должны быть известны субъектный предел применимости плоской модели,
  абсолютная измеримость и геометрия на КТ/FEM.
- Разные семейства могут давать разные ответы, потому что проверяют разные
  свойства. Их нельзя голосованием превращать в один универсальный $L_{min}$.
- При безразмерном критерии и фиксированных $\rho_2/\rho_1,\beta$ непрерывный
  модельный $L_{min}$ масштабируется с $h$ (`30.05`). Дискретный изготовленный
  ряд округляет этот закон вверх и может вообще не содержать проходящего
  кандидата.
- Абсолютный порог не обладает таким масштабным законом при фиксированном
  пороге в омах. Его значение должно приходить из конкретной измерительной
  задачи, а не из прямой модели.


## Условия определения реального $L_{min}$

Для каждого прибора и добровольца до численного вывода нужны:

1. принятый после QC субъектный список размеров;
2. калиброванный оператор наблюдения, gain/offset, знак/модуль и масштаб
   переменного канала;
3. заранее заданная целевая величина: обнаружение $\Delta Z$, оценка
   $\Delta\rho_2$ или предварительный фильтр перед выбором пары;
4. допустимая ошибка этой величины и количественный бюджет `31.04`;
5. $h$ из КТ для полной разработочной постановки либо отдельно валидированная
   замена $h$;
6. допустимое множество размеров по индивидуальному КТ/FEM-критерию `20.12`;
7. проверка результата на данных, включая фиксированный порядок 140→50,
   последовательность записей и субъектный дубликат.

До этого проектный статус:

```text
Lmin_status = not_determined
reason = thresholds_calibration_uncertainty_and_subject_specific_Lmax_missing
```

Следующий документ `32.02` рассматривает пары при известной $h$ и не должен
принимать условный результат этого notebook за готовое ограничение.
